# Part 5: `test_input_sequence1.mp4` Colab GPU 자동 라벨링 (`YOLO11x` 0°/180° 회전 보정)
- 런타임 유형: **GPU (T4/L4/A100)**
- 머리→발 시점(거꾸로 누운 자세) 보정을 위해 `0°` 원본과 `180°` 회전 입력을 모두 추론해 더 높은 신뢰도/관절 품질 결과를 선택하고 좌표를 원본 좌표계로 복원합니다.
- 출력: `images/`, `labels_bbox/`, `labels_seg/`, `labels_pose/`, `annotations_coco.json`, `frame_labels.csv` -> `test_input_sequence1_labeled.zip` 다운로드


In [ ]:
!nvidia-smi
!pip -q install ultralytics opencv-python-headless numpy


In [ ]:
# 방법 A: 로컬에서 test_input_sequence1.mp4 직접 업로드
from google.colab import files
uploaded = files.upload()
VIDEO_PATH = next(iter(uploaded.keys()))
print("Uploaded:", VIDEO_PATH)

# 방법 B: Google Drive 사용 시 위 주석 처리 후 아래 사용
# from google.colab import drive
# drive.mount("/content/drive")
# VIDEO_PATH = "/content/drive/MyDrive/test_input_sequence1.mp4"


In [ ]:
from pathlib import Path

ROI_CFG = '{\n  "roi_version": "test_input_sequence1_perspective_trapezoid_v1",\n  "roi_source": "frame_0000 supine baseline of test_input_sequence1.mp4; board edges extended along perspective lines toward foot side",\n  "roi_reviewed": true,\n  "deployment_approved": false,\n  "bed_polygon": [\n    [0.278, 0.040],\n    [0.626, 0.040],\n    [0.882, 0.970],\n    [0.073, 0.970]\n  ],\n  "raw_board_polygon": [\n    [0.255, 0.145],\n    [0.655, 0.150],\n    [0.880, 0.962],\n    [0.075, 0.962]\n  ],\n  "perspective_extension": {\n    "far_edge_ratio": 0.1285,\n    "direction": "head_to_foot_upward",\n    "reason": "Camera views from head toward feet; feet extend past raw board top edge in image space."\n  }\n}\n'
TIMELINE_CFG = '{\n  "video": "test_input_sequence1.mp4",\n  "fps": 30.0,\n  "total_frames": 1024,\n  "roi_config": "test_input_sequence1_roi_v1.json",\n  "verification_note": "Visually verified at 15-frame (0.5s) resolution against test_input_sequence1_roi_v1.json bed_polygon.",\n  "events": [\n    {"start_sec": 0.0, "end_sec": 0.75, "start_frame": 0, "end_frame": 23, "exit_hint": false, "phase": "supine_baseline", "description": "정자세 (ROI 내부)"},\n    {"start_sec": 0.75, "end_sec": 3.25, "start_frame": 23, "end_frame": 98, "exit_hint": true, "phase": "left_side_arm_exit", "description": "화면 왼쪽으로 오른팔 뻗어 ROI 밖 이탈"},\n    {"start_sec": 3.25, "end_sec": 4.75, "start_frame": 98, "end_frame": 143, "exit_hint": false, "phase": "supine_return_1", "description": "정자세 복귀 (ROI 내부)"},\n    {"start_sec": 4.75, "end_sec": 6.25, "start_frame": 143, "end_frame": 188, "exit_hint": true, "phase": "right_side_arm_exit", "description": "화면 오른쪽으로 왼팔 뻗어 ROI 밖 이탈"},\n    {"start_sec": 6.25, "end_sec": 7.25, "start_frame": 188, "end_frame": 218, "exit_hint": false, "phase": "supine_return_2", "description": "정자세 복귀 (ROI 내부)"},\n    {"start_sec": 7.25, "end_sec": 8.75, "start_frame": 218, "end_frame": 263, "exit_hint": true, "phase": "all_limbs_exit", "description": "양팔·양다리 대자로 ROI 밖 이탈"},\n    {"start_sec": 8.75, "end_sec": 10.25, "start_frame": 263, "end_frame": 308, "exit_hint": false, "phase": "supine_return_3", "description": "정자세 복귀 (ROI 내부)"},\n    {"start_sec": 10.25, "end_sec": 13.25, "start_frame": 308, "end_frame": 398, "exit_hint": true, "phase": "left_roll_shift_exit", "description": "화면 왼쪽으로 몸통·다리 쏠리며 ROI 왼쪽 경계 이탈"},\n    {"start_sec": 13.25, "end_sec": 14.75, "start_frame": 398, "end_frame": 443, "exit_hint": false, "phase": "supine_safe", "description": "정자세 (ROI 내부)"},\n    {"start_sec": 14.75, "end_sec": 17.25, "start_frame": 443, "end_frame": 518, "exit_hint": true, "phase": "right_side_leg_arm_exit", "description": "화면 오른쪽으로 다리·팔 ROI 밖 이탈"},\n    {"start_sec": 17.25, "end_sec": 18.75, "start_frame": 518, "end_frame": 563, "exit_hint": false, "phase": "sit_up_inside", "description": "일어나 앉는 자세 (상체 ROI 내부)"},\n    {"start_sec": 18.75, "end_sec": 21.75, "start_frame": 563, "end_frame": 653, "exit_hint": true, "phase": "sit_up_top_exit", "description": "앉은 상태에서 머리·상체가 상단 ROI 경계 밖 이탈"},\n    {"start_sec": 21.75, "end_sec": 22.75, "start_frame": 653, "end_frame": 683, "exit_hint": false, "phase": "lie_back_inside", "description": "다시 누우며 ROI 내부 복귀"},\n    {"start_sec": 22.75, "end_sec": 24.25, "start_frame": 683, "end_frame": 728, "exit_hint": true, "phase": "turn_left_edge_exit", "description": "돌아눕는 과정에서 팔/어깨가 왼쪽 ROI 경계 밖 이탈"},\n    {"start_sec": 24.25, "end_sec": 25.75, "start_frame": 728, "end_frame": 773, "exit_hint": false, "phase": "prone_inside", "description": "엎드린 상태 (팔꿈치 포함 ROI 내부)"},\n    {"start_sec": 25.75, "end_sec": 28.25, "start_frame": 773, "end_frame": 848, "exit_hint": true, "phase": "prone_arms_exit", "description": "엎드린 채 양팔·한팔 좌우 ROI 밖 이탈"},\n    {"start_sec": 28.25, "end_sec": 29.75, "start_frame": 848, "end_frame": 893, "exit_hint": false, "phase": "pushup_inside", "description": "양팔 모으고 상체 일으키기 직전 (ROI 내부)"},\n    {"start_sec": 29.75, "end_sec": 34.13, "start_frame": 893, "end_frame": 1024, "exit_hint": true, "phase": "stand_up_exit", "description": "상체 상단 ROI 이탈 후 기립 및 보행 이탈"}\n  ]\n}\n'

Path("roi.json").write_text(ROI_CFG, encoding="utf-8")
Path("timeline.json").write_text(TIMELINE_CFG, encoding="utf-8")
print("Saved roi.json and timeline.json")


In [ ]:
import csv, json
from pathlib import Path
import cv2
import numpy as np
import torch
from ultralytics import YOLO

STEP = 1
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
OUT_DIR = Path("test_input_sequence1_labeled")
print("Device:", DEVICE)

roi_pts = np.asarray(json.loads(Path("roi.json").read_text())["bed_polygon"], dtype=np.float32)
timeline_events = json.loads(Path("timeline.json").read_text())["events"]

COCO_COMMON = (0, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16)
TORSO = (5, 6, 11, 12)

def lookup_timeline(frame_idx):
    for ev in timeline_events:
        if ev["start_frame"] <= frame_idx < ev["end_frame"]:
            hint = "EXIT" if ev["exit_hint"] is True else ("SAFE" if ev["exit_hint"] is False else "END")
            return ev["phase"], hint
    return "unknown", "UNKNOWN"

def roi_contains(pt, shape, tol=3.0):
    h, w = shape[:2]
    poly = roi_pts * np.array([w - 1, h - 1], np.float32)
    return cv2.pointPolygonTest(poly, (float(pt[0]), float(pt[1])), True) >= -tol

def pose_exit_hint(pts, shape):
    usable = np.isfinite(pts).all(axis=1) & (pts[:, 2] >= 0.5)
    if sum(usable[i] for i in TORSO) < 2:
        return "UNAVAIL"
    sel = pts[[i for i in COCO_COMMON if usable[i]], :2]
    if any(not roi_contains(p, shape) for p in sel):
        return "EXIT"
    if not all(usable[i] for i in COCO_COMMON):
        return "UNAVAIL"
    return "SAFE"

def pick_best_det(model, frame, frame_180):
    best = (0.0, False, None)
    for rotated, img in ((False, frame), (True, frame_180)):
        res = model.predict(source=img, classes=[0], conf=0.5, imgsz=640, device=DEVICE, verbose=False)[0]
        if res.boxes is not None and len(res.boxes) > 0:
            score = float(res.boxes.conf.max().cpu().item())
            if score > best[0]:
                best = (score, rotated, res)
    return best[1], best[2]

def pick_best_pose(model, frame, frame_180):
    best = (-1.0, False, None)
    for rotated, img in ((False, frame), (True, frame_180)):
        res = model.predict(source=img, classes=[0], conf=0.5, imgsz=640, device=DEVICE, verbose=False)[0]
        if res.boxes is not None and len(res.boxes) > 0 and res.keypoints is not None:
            kconf = res.keypoints.conf.cpu().numpy()[0]
            vis_common = float(sum(kconf[i] >= 0.5 for i in COCO_COMMON))
            torso_mean = float(kconf[list(TORSO)].mean())
            score = vis_common + torso_mean
            if score > best[0]:
                best = (score, rotated, res)
    return best[1], best[2]

for sub in ("images", "labels_bbox", "labels_seg", "labels_pose"):
    (OUT_DIR / sub).mkdir(parents=True, exist_ok=True)

det_yolo = YOLO("yolo11x.pt")
seg_yolo = YOLO("yolo11x-seg.pt")
pose_yolo = YOLO("yolo11x-pose.pt")

cap = cv2.VideoCapture(VIDEO_PATH)
fps = float(cap.get(cv2.CAP_PROP_FPS) or 30.0)
coco = {"images": [], "annotations": [], "categories": [{"id": 1, "name": "person", "supercategory": "person"}]}
csv_rows, ann_id, frame_idx, saved = [], 1, 0, 0

while True:
    ok, frame = cap.read()
    if not ok or frame is None:
        break
    if frame_idx % STEP != 0:
        frame_idx += 1
        continue
    h, w = frame.shape[:2]
    frame_180 = cv2.rotate(frame, cv2.ROTATE_180)
    stem = f"frame_{frame_idx:04d}"
    img_name = f"{stem}.jpg"
    cv2.imwrite(str(OUT_DIR / "images" / img_name), frame)

    # 1) BBox (0 vs 180 deg)
    det_rot, det_res = pick_best_det(det_yolo, frame, frame_180)
    bbox_lines, det_exit = [], "UNAVAIL"
    if det_res is not None:
        for xywhn in det_res.boxes.xywhn.cpu().numpy():
            cx, cy = (1.0 - xywhn[0], 1.0 - xywhn[1]) if det_rot else (xywhn[0], xywhn[1])
            bbox_lines.append(f"0 {cx:.6f} {cy:.6f} {xywhn[2]:.6f} {xywhn[3]:.6f}")
        b0 = det_res.boxes.xyxy.cpu().numpy()[0]
        if det_rot:
            b0 = np.array([w - b0[2], h - b0[3], w - b0[0], h - b0[1]], dtype=np.float32)
        det_exit = "SAFE" if roi_contains(((b0[0]+b0[2])/2, (b0[1]+b0[3])/2), frame.shape, 0.0) else "EXIT"
    (OUT_DIR / "labels_bbox" / f"{stem}.txt").write_text("\n".join(bbox_lines), encoding="utf-8")

    # 2) Segmentation (0 vs 180 deg)
    seg_rot, seg_res = pick_best_det(seg_yolo, frame, frame_180)
    seg_lines = []
    if seg_res is not None and seg_res.masks is not None and seg_res.masks.xyn is not None:
        for poly in seg_res.masks.xyn:
            if len(poly) >= 3:
                pts = (1.0 - poly) if seg_rot else poly
                seg_lines.append("0 " + " ".join(f"{pt[0]:.6f} {pt[1]:.6f}" for pt in pts))
    (OUT_DIR / "labels_seg" / f"{stem}.txt").write_text("\n".join(seg_lines), encoding="utf-8")

    # 3) Pose (0 vs 180 deg)
    pose_rot, pose_res = pick_best_pose(pose_yolo, frame, frame_180)
    pose_lines, pose_exit = [], "UNAVAIL"
    coco["images"].append({"id": frame_idx + 1, "file_name": img_name, "width": w, "height": h})
    if pose_res is not None:
        xywhn_all = pose_res.boxes.xywhn.cpu().numpy()
        xyxy_all = pose_res.boxes.xyxy.cpu().numpy()
        kpts_xy = pose_res.keypoints.xy.cpu().numpy()
        kpts_xyn = pose_res.keypoints.xyn.cpu().numpy()
        kpts_conf = pose_res.keypoints.conf.cpu().numpy()
        if pose_rot:
            xywhn_all[:, 0] = 1.0 - xywhn_all[:, 0]
            xywhn_all[:, 1] = 1.0 - xywhn_all[:, 1]
            xyxy_all = np.stack([w - xyxy_all[:, 2], h - xyxy_all[:, 3], w - xyxy_all[:, 0], h - xyxy_all[:, 1]], axis=1)
            kpts_xy[:, :, 0] = w - 1 - kpts_xy[:, :, 0]
            kpts_xy[:, :, 1] = h - 1 - kpts_xy[:, :, 1]
            kpts_xyn[:, :, 0] = 1.0 - kpts_xyn[:, :, 0]
            kpts_xyn[:, :, 1] = 1.0 - kpts_xyn[:, :, 1]
        for i in range(len(xywhn_all)):
            bx, by, bw, bh = (float(v) for v in xywhn_all[i])
            kp_tokens, coco_kps = [], []
            for k in range(17):
                kx, ky, kc = float(kpts_xyn[i, k, 0]), float(kpts_xyn[i, k, 1]), float(kpts_conf[i, k])
                vis = 2 if kc >= 0.5 else (1 if kc > 0.1 else 0)
                kp_tokens.append(f"{kx:.6f} {ky:.6f} {vis}")
                coco_kps.extend([float(kpts_xy[i, k, 0]), float(kpts_xy[i, k, 1]), vis])
            pose_lines.append(f"0 {bx:.6f} {by:.6f} {bw:.6f} {bh:.6f} " + " ".join(kp_tokens))
            x1, y1, x2, y2 = (float(v) for v in xyxy_all[i])
            coco["annotations"].append({
                "id": ann_id, "image_id": frame_idx + 1, "category_id": 1,
                "bbox": [x1, y1, max(0.0, x2-x1), max(0.0, y2-y1)],
                "area": max(0.0, (x2-x1)*(y2-y1)), "iscrowd": 0,
                "num_keypoints": sum(1 for v in coco_kps[2::3] if v > 0), "keypoints": coco_kps,
            })
            ann_id += 1
        pose_exit = pose_exit_hint(np.concatenate([kpts_xy[0], kpts_conf[0][:, None]], axis=1), frame.shape)
    (OUT_DIR / "labels_pose" / f"{stem}.txt").write_text("\n".join(pose_lines), encoding="utf-8")

    phase, tl_hint = lookup_timeline(frame_idx)
    csv_rows.append({
        "frame_idx": frame_idx, "image": img_name, "timestamp_sec": f"{frame_idx / fps:.3f}",
        "timeline_phase": phase, "timeline_exit_hint": tl_hint,
        "auto_exit_det_center": det_exit, "auto_exit_pose": pose_exit,
        "pose_rotated_180": pose_rot,
    })
    frame_idx += 1
    saved += 1

cap.release()
(OUT_DIR / "annotations_coco.json").write_text(json.dumps(coco, indent=2), encoding="utf-8")
with (OUT_DIR / "frame_labels.csv").open("w", newline="", encoding="utf-8") as f:
    w_csv = csv.DictWriter(f, fieldnames=list(csv_rows[0].keys()))
    w_csv.writeheader()
    w_csv.writerows(csv_rows)
print(f"Done! Labeled {saved} frames in {OUT_DIR} (180-deg pose selected: {sum(r['pose_rotated_180'] for r in csv_rows)}/{saved})")


In [ ]:
!zip -qr test_input_sequence1_labeled.zip test_input_sequence1_labeled/
from google.colab import files
files.download("test_input_sequence1_labeled.zip")
